# Section 7, Test 1: Real Generative DDPM, Local Rule vs. Backprop vs. Two-Factor Hebbian

**What this is, and what it isn't.** This is the first genuine head-to-head on the REAL generative
diffusion process (actual reverse sampling producing images), not the reconstruction-proxy task used
elsewhere. All three methods -- backprop, the three-factor local rule, and the two-factor Hebbian
rule -- train the SAME convolutional U-Net denoiser architecture (base_ch=32, P=351,585, the
validated architecture from the foundation notebook), on the SAME real MNIST data, under the SAME
DDPM noise schedule (T=1000, the confirmed fix).

**A real theoretical caveat, stated up front rather than discovered after a disappointing result:**
the three-factor rule as defined (Eq. 1 in the paper) broadcasts ONE global scalar to every parameter
(K=1), regardless of architecture. Diffusion's theoretical advantage (coupling scope S=1) requires
per-channel or per-mode feedback to actually pay off -- a single global scalar does not automatically
inherit that advantage just because the underlying loss is separable. So this test honestly measures
the PLAIN rule on this architecture, and per the barrier law, is expected to hit the same P-scale wall
the MLP did (M* ~ P/3 ~ 117,000 probes at P=351,585). If it fails decisively, that is a real,
theory-predicted finding -- not a bug, and not a reason to quietly change the rule to look better.

**Equal compute budget, same discipline as the MLP Test 1.** Backprop trains for a fixed number of
steps; the three-factor rule gets a step budget matched in total forward-pass-equivalents, not in
step count -- exactly the same protocol used for the MLP's Test 1.

**Two-factor Hebbian on a conv net -- a real design choice, documented rather than hidden.** The
classical rule (Δθ ∝ pre × post) is defined for a single linear layer. For a conv layer, we generalize
it via the standard im2col/unfold trick: extract input patches at each spatial position ("pre"),
correlate with the output feature map at that position ("post"), and average over batch and spatial
location to get a weight-shaped update. This is a legitimate, standard generalization, but it is a
choice -- not a canonical, unambiguous extension -- and is called out explicitly here.

**The real check, same as the foundation notebook:** actual reverse-sampling generation for each
trained model, not just a loss number.

## Step 0 — Setup

In [ ]:
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('WARNING: this will be very slow on CPU. Use a GPU runtime.')

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS_LOG = []

def log_result(name, config, result, seed):
    entry = {'name': name, 'seed': seed, 'config': config, 'result': result,
              'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    RESULTS_LOG.append(entry)
    print(f"[logged] {name}: {result}")
    return entry

def save_provenance(path='provenance_diffusion_test1.json'):
    with open(path, 'w') as f:
        json.dump(RESULTS_LOG, f, indent=2, default=str)
    print(f'Saved {len(RESULTS_LOG)} logged results to {path}')


## Step 1 — Noise schedule and architecture (identical to the validated foundation)

In [ ]:
T_STEPS = 1000
beta_start, beta_end = 1e-4, 0.02
betas = torch.linspace(beta_start, beta_end, T_STEPS, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def forward_diffusion(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ab = alpha_bars[t].sqrt().view(-1, 1, 1, 1)
    sqrt_1mab = (1 - alpha_bars[t]).sqrt().view(-1, 1, 1, 1)
    return sqrt_ab * x0 + sqrt_1mab * noise, noise

assert alpha_bars[-1].item() < 0.01, 'Schedule does not reach near-pure noise.'
print(f'T={T_STEPS}, alpha_bar[-1]={alpha_bars[-1].item():.6f} -- OK')

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act = nn.SiLU()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.act(self.norm2(self.conv2(h)))
        return h

class SmallUNet(nn.Module):
    def __init__(self, base_ch=32, time_dim=32):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim)
        )
        self.enc1 = ConvBlock(1, base_ch, time_dim)
        self.down1 = nn.Conv2d(base_ch, base_ch, 3, stride=2, padding=1)
        self.enc2 = ConvBlock(base_ch, base_ch * 2, time_dim)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 3, stride=2, padding=1)
        self.bottleneck = ConvBlock(base_ch * 2, base_ch * 2, time_dim)
        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.dec2 = ConvBlock(base_ch * 4, base_ch, time_dim)
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.dec1 = ConvBlock(base_ch * 2, base_ch, time_dim)
        self.out_conv = nn.Conv2d(base_ch, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        e1 = self.enc1(x, t_emb)
        e2 = self.enc2(self.down1(e1), t_emb)
        b = self.bottleneck(self.down2(e2), t_emb)
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1), t_emb)
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1), t_emb)
        return self.out_conv(d1)

def make_denoiser(seed=SEED):
    torch.manual_seed(seed)
    return SmallUNet(base_ch=32, time_dim=32).to(device)

P_denoiser = sum(p.numel() for p in make_denoiser().parameters())
print(f'Denoiser parameter count: P = {P_denoiser:,}')


## Step 2 — Real MNIST data and the validated x0-clipping sampler

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1),
])
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

N_TRAIN = 8000
train_idx = torch.randperm(len(mnist_train))[:N_TRAIN]
X_train = torch.stack([mnist_train[i][0] for i in train_idx]).to(device)
print(f'Training images: {X_train.shape}')

@torch.no_grad()
def sample_ddpm(model, n_samples=8, seed=None):
    model.eval()
    if seed is not None:
        gen = torch.Generator(device=device).manual_seed(seed)
        x = torch.randn(n_samples, 1, 28, 28, device=device, generator=gen)
    else:
        x = torch.randn(n_samples, 1, 28, 28, device=device)
    for t_step in reversed(range(T_STEPS)):
        t_batch = torch.full((n_samples,), t_step, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)
        alpha_t = alphas[t_step]; alpha_bar_t = alpha_bars[t_step]
        alpha_bar_prev = alpha_bars[t_step - 1] if t_step > 0 else torch.tensor(1.0, device=device)
        beta_t = betas[t_step]
        x0_pred = ((x - (1 - alpha_bar_t).sqrt() * pred_noise) / alpha_bar_t.sqrt()).clamp(-1.0, 1.0)
        posterior_mean = (
            (alpha_bar_prev.sqrt() * beta_t / (1 - alpha_bar_t)) * x0_pred
            + (alpha_t.sqrt() * (1 - alpha_bar_prev) / (1 - alpha_bar_t)) * x
        )
        posterior_var = beta_t * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
        x = posterior_mean + posterior_var.sqrt() * torch.randn_like(x) if t_step > 0 else posterior_mean
    model.train()
    return x

def show_sample_grid(samples, title, save_path):
    disp = (samples.clamp(-1, 1) + 1) / 2
    fig, axes = plt.subplots(1, 8, figsize=(16, 2))
    for i, ax in enumerate(axes):
        ax.imshow(disp[i, 0].cpu().numpy(), cmap='gray'); ax.axis('off')
    plt.suptitle(title); plt.savefig(save_path, dpi=120, bbox_inches='tight'); plt.show()


## Step 3 — Method A: Backprop (the reference, matched step budget)

Trained for a modest, fixed step budget (this is a from-scratch Test 1 comparison, not the longer
foundation-quality run) -- backprop's total forward-pass-equivalent budget is what the three-factor
rule below gets matched against.

In [ ]:
BATCH_SIZE = 64
BP_STEPS = 500
LR_BP = 2e-4

denoiser_bp = make_denoiser(SEED)
opt = torch.optim.Adam(denoiser_bp.parameters(), lr=LR_BP)
losses_bp = []
t0 = time.time()
for step in range(BP_STEPS):
    idx = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), device=device)
    x0 = X_train[idx]
    t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=device)
    x_t, noise = forward_diffusion(x0, t)
    opt.zero_grad(set_to_none=True)
    pred_noise = denoiser_bp(x_t, t)
    loss = F.mse_loss(pred_noise, noise)
    loss.backward()
    opt.step()
    losses_bp.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f'backprop step {step+1}/{BP_STEPS}  loss={np.mean(losses_bp[-100:]):.4f}')

bp_time = time.time() - t0
bp_forward_pass_equiv = BP_STEPS * 2   # 1 forward + 1 backward per step
print(f'\nBackprop done in {bp_time:.1f}s, final loss {np.mean(losses_bp[-50:]):.4f}')
print(f'Backprop forward-pass-equivalent budget: {bp_forward_pass_equiv}')
log_result('diffusion_test1_backprop', {'steps': BP_STEPS, 'lr': LR_BP},
           {'final_loss': float(np.mean(losses_bp[-50:])), 'wall_clock_s': bp_time,
            'forward_pass_equiv_budget': bp_forward_pass_equiv}, SEED)

samples_bp = sample_ddpm(denoiser_bp, n_samples=8, seed=777)
show_sample_grid(samples_bp, 'Backprop, from scratch (matched budget)', 'diffusion_test1_backprop_samples.png')


## Step 4 — Method B: Three-factor local rule (equal forward-pass-equivalent budget)

$M$ antithetic probe pairs per step, applied to the FULL flattened parameter vector (the plain,
global-scalar rule, $K=1$, as flagged in the introduction to this notebook).

In [ ]:
def flat_params(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def set_flat_params(model, flat):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat[offset:offset+n].view_as(p))
        offset += n

def diffusion_loss_at(model, flat, x0, t, noise):
    set_flat_params(model, flat)
    with torch.no_grad():
        x_t, _ = forward_diffusion(x0, t, noise)
        pred_noise = model(x_t, t)
        return F.mse_loss(pred_noise, noise).item()

M_PROBES = 20
SIGMA = 0.01
LR_LOCAL = 5e-3

local_steps_budget = bp_forward_pass_equiv // (2 * M_PROBES)
print(f'Three-factor step budget at M={M_PROBES}: {local_steps_budget} steps '
      f'(uses {local_steps_budget*2*M_PROBES} forward-pass-equivalents, vs backprop\'s {bp_forward_pass_equiv})')

denoiser_local = make_denoiser(SEED)
theta = flat_params(denoiser_local)
D = theta.numel()
losses_local = []
t0 = time.time()
for step in range(local_steps_budget):
    idx = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), device=device)
    x0 = X_train[idx]
    t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=device)
    noise = torch.randn_like(x0)
    g_hat = torch.zeros_like(theta)
    for _ in range(M_PROBES):
        xi = torch.randn(D, device=device)
        l_plus = diffusion_loss_at(denoiser_local, theta + SIGMA * xi, x0, t, noise)
        l_minus = diffusion_loss_at(denoiser_local, theta - SIGMA * xi, x0, t, noise)
        g_hat += xi * (l_plus - l_minus) / (2 * SIGMA)
    g_hat /= M_PROBES
    theta = theta - LR_LOCAL * g_hat
    step_loss = diffusion_loss_at(denoiser_local, theta, x0, t, noise)
    losses_local.append(step_loss)
    if (step + 1) % max(1, local_steps_budget // 10) == 0:
        print(f'three-factor step {step+1}/{local_steps_budget}  loss={step_loss:.4f}')

set_flat_params(denoiser_local, theta)
local_time = time.time() - t0
print(f'\nThree-factor done in {local_time:.1f}s, final loss {losses_local[-1]:.4f}')
log_result('diffusion_test1_three_factor', {'steps': local_steps_budget, 'M': M_PROBES, 'lr': LR_LOCAL, 'sigma': SIGMA,
                                              'param_count': D},
           {'final_loss': losses_local[-1], 'wall_clock_s': local_time}, SEED)

samples_local = sample_ddpm(denoiser_local, n_samples=8, seed=777)
show_sample_grid(samples_local, 'Three-factor local rule, from scratch (matched budget)', 'diffusion_test1_local_samples.png')


## Step 5 — Method C: Two-factor Hebbian (generalized to conv layers via im2col)

For each conv layer, extract input patches at every spatial position ("pre"), correlate with the
output feature map at that position ("post"), and average over batch and space to get a
weight-shaped update -- the standard generalization of $\Delta\theta \propto \text{pre}\times\text{post}$
to convolutions.

In [ ]:
def hebbian_conv_update(conv_layer, x_in, x_out, eta):
    """x_in: [B, Cin, H, W] input to the conv; x_out: [B, Cout, H', W'] its output."""
    kh, kw = conv_layer.kernel_size
    stride = conv_layer.stride[0]
    padding = conv_layer.padding[0]
    patches = F.unfold(x_in, kernel_size=(kh, kw), stride=stride, padding=padding)   # [B, Cin*kh*kw, L]
    B, _, L = patches.shape
    out_flat = x_out.reshape(B, x_out.shape[1], -1)   # [B, Cout, L]
    # correlate patches with output at each spatial location, average over batch and location
    update = torch.einsum('bpl,bol->op', patches, out_flat) / (B * L)   # [Cout, Cin*kh*kw]
    update = update.view(conv_layer.out_channels, conv_layer.in_channels, kh, kw)
    conv_layer.weight.data += eta * update

ETA_HEB = 1e-6   # small, since conv correlation magnitudes differ from the MLP case
HEB_STEPS = local_steps_budget   # same step count as the three-factor rule, for a fair three-way comparison

denoiser_heb = make_denoiser(SEED)
conv_layers = [m for m in denoiser_heb.modules() if isinstance(m, nn.Conv2d)]

activity = {}
def hook_factory(key):
    def hook(module, inp, out):
        activity[key] = (inp[0].detach(), out.detach())
    return hook
handles = [layer.register_forward_hook(hook_factory(i)) for i, layer in enumerate(conv_layers)]

losses_heb = []
t0 = time.time()
for step in range(HEB_STEPS):
    idx = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), device=device)
    x0 = X_train[idx]
    t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=device)
    with torch.no_grad():
        x_t, noise = forward_diffusion(x0, t)
        pred_noise = denoiser_heb(x_t, t)
        loss = F.mse_loss(pred_noise, noise)
    for i, layer in enumerate(conv_layers):
        x_in, x_out = activity[i]
        hebbian_conv_update(layer, x_in, x_out, ETA_HEB)
    losses_heb.append(loss.item())
    if (step + 1) % max(1, HEB_STEPS // 10) == 0:
        print(f'two-factor step {step+1}/{HEB_STEPS}  loss={loss.item():.4f}')

for h in handles:
    h.remove()
heb_time = time.time() - t0
print(f'\nTwo-factor done in {heb_time:.1f}s, final loss {losses_heb[-1]:.4f}')
log_result('diffusion_test1_two_factor', {'steps': HEB_STEPS, 'eta': ETA_HEB},
           {'final_loss': losses_heb[-1], 'wall_clock_s': heb_time}, SEED)

samples_heb = sample_ddpm(denoiser_heb, n_samples=8, seed=777)
show_sample_grid(samples_heb, 'Two-factor Hebbian, from scratch (matched budget)', 'diffusion_test1_two_factor_samples.png')


## Step 6 — Head-to-head summary

In [ ]:
print(f'{"Method":<20}{"Final loss":>15}{"Steps":>10}{"Wall-clock (s)":>18}')
print('-' * 63)
print(f'{"Backprop":<20}{losses_bp[-1]:>15.4f}{BP_STEPS:>10}{bp_time:>18.1f}')
print(f'{"Three-factor":<20}{losses_local[-1]:>15.4f}{local_steps_budget:>10}{local_time:>18.1f}')
print(f'{"Two-factor Hebbian":<20}{losses_heb[-1]:>15.4f}{HEB_STEPS:>10}{heb_time:>18.1f}')

log_result('diffusion_test1_summary', {},
           {'backprop_final_loss': losses_bp[-1], 'three_factor_final_loss': losses_local[-1],
            'two_factor_final_loss': losses_heb[-1]}, SEED)

print()
print('THE REAL CHECK: compare the three sample grids above directly.')
print('Given the barrier law at P=351,585, the three-factor rule is expected to fail decisively')
print('under this equal-compute budget (M*~P/3~117K probes for meaningful alignment, vs the')
print(f'{M_PROBES}-probe budget actually used) -- if the samples look like noise/blobs rather than')
print('digits, that CONFIRMS the theory rather than indicating a bug. Report whatever you see.')


## Step 7 — Save provenance

In [ ]:
save_provenance('provenance_diffusion_test1.json')
print()
for entry in RESULTS_LOG:
    print(f"  - {entry['name']}: {entry['result']}")


## Summary

This is the first real generative (not reconstruction) head-to-head for the diffusion track,
matching the equal-compute-budget rigor already used for the MLP's Test 1. Two things to report
honestly once run:

1. **The actual sample grids** -- this is the real test, not the loss numbers alone (per the lesson
   learned building the foundation notebook).
2. **Whether the three-factor rule fails at this scale**, as the barrier law predicts for a plain,
   global-scalar (K=1) rule at P=351,585. A decisive failure here is not a bug -- it would be the
   same kind of honest, theory-confirming negative result as the MLP's Test 1, and worth reporting
   exactly that way rather than tuning hyperparameters until it looks better.

The two-factor Hebbian generalization to conv layers (im2col-based) is a real design choice made
explicit in this notebook -- flag it in the writeup as a choice, not a canonical extension, since
Cameron's version may define this differently if you merge.